<div style="background-color: #2ccece; padding: 20px; border-radius: 10px;">

<h1 style="color: black;">

NLP Project


</div>


In [14]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

## Step 1: Exploration

In [5]:
# Path to the reviews dataset
REVIEWS_PATH = r"data\TripAdvisorTrainingDataProject1\reviews83325.csv"
reviews = pd.read_csv(REVIEWS_PATH)

# Dataset overview
print("Shape:", reviews.shape)
print("Columns:", reviews.columns.tolist())
display(reviews.head(5))



C:\Users\sandr\AppData\Local\Temp\ipykernel_10784\1399121962.py:3: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(REVIEWS_PATH)


Shape: (340385, 21)
Columns: ['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title']


,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,758953508,188467,A Treasure in Paris,9262311F3378F8CC4709DD4D92380278,We come back to this huge square every time we...,5,5/7/2020 03:16:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,755705747,188467,A place to take a breath from exploring Paris.,836BAA8786B81033412B950CF5BDA70C,The most beautiful square in Paris has a stran...,5,31/5/2020 19:36:17,2019-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,750396525,188467,Beautiful and elegant,8FEC7B1C9034428EFF3BF8728B036829,"Lovely architecture, looks different again to ...",4,11/3/2020 06:16:38,2020-03,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Random sample of reviews
display(reviews.sample(5, random_state=42))

,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
105478,127684269,292257,HIP HIP MARAIS....,DFA5F34122ECCC4687BE89DA025BF4D0,we spent a super day wandering this beautiful ...,5,11/4/2012 16:55:03,2012-04,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
312069,560967760,11459530,Expérience captivante,3D45A9C4263692474F29AC1BB578507F,Une super expérience: notre guide néerlandaise...,5,17/2/2018 05:19:20,2018-02,fr,Desktop,...,[],False,False,561402920.0,fr,19/2/2018 04:49:17,Responsable relations clients,Cultival,"Cher visiteur, \nNous sommes ravis que vous ay...",Owner Response
50928,286951586,188679,Must see epic and mysterious,CF1F9E57DF1CE41C73BE079901C32145,This cathedral is a world wonder the architec...,5,8/7/2015 20:13:42,2014-08,en,Mobile,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
161184,658072350,718765,Superbe,D6EEC8C74AF03E79C423E9FF9C7798BF,Léon est un très bon restaurant. Chaque fois q...,5,12/3/2019 14:51:19,2019-03,fr,Desktop,...,"[{""name"":""Service"",""value"":""5""},{""name"":""Value...",False,False,658298558.0,fr,13/3/2019 16:31:42,Responsable relations clients,Léon d,C'est un plaisir de vous faire plaisir :) A bi...,Owner Response
231041,385772697,2176008,Wonderful crepes,EF97218BDDA9231A64E2095EF78CB173,Our last lunch before leaving for the airport....,5,25/6/2016 10:12:54,2016-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# mIssing values
print("\nMissing values:")
print(reviews.isna().sum().sort_values(ascending=False).head(20))



Missing values:
owner_responder         300988
owner_connection        300745
owner_title             300744
owner_response          300744
owner_date_review       300744
owner_langue            300744
owner_id                300744
date_visit               19238
idauteur                   659
titre                        3
machine_translated           0
machine_translatable         0
id                           0
subratings                   0
idplace                      0
published_platform           0
langue                       0
date_review                  0
note                         0
review                       0
dtype: int64


In [9]:
# Distribution of review languages
if "langue" in reviews.columns:
    display(reviews["langue"].value_counts().head(10))

langue
en    153071
fr     99078
it     22912
es     21768
pt     19521
ru      5463
de      5237
ja      4124
nl      2835
ko      1133
Name: count, dtype: int64

## Step 2: Text preprocessing

1) Filter the reviews: english only

In [10]:

LANG_COL = "langue"
TEXT_COL = "review"
PLACE_ID_COL = "idplace"

# Filtrer in egnlish
reviews_en = reviews.copy()
if LANG_COL in reviews_en.columns:
    reviews_en = reviews_en[reviews_en[LANG_COL].astype(str).str.lower().eq("en")]

# Keep the rows with non-missing text and place_id
reviews_en = reviews_en.dropna(subset=[TEXT_COL, PLACE_ID_COL])
reviews_en[TEXT_COL] = reviews_en[TEXT_COL].astype(str)

print("English reviews shape:", reviews_en.shape)
display(reviews_en.head(5))

English reviews shape: (153071, 21)


,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,758953508,188467,A Treasure in Paris,9262311F3378F8CC4709DD4D92380278,We come back to this huge square every time we...,5,5/7/2020 03:16:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,755705747,188467,A place to take a breath from exploring Paris.,836BAA8786B81033412B950CF5BDA70C,The most beautiful square in Paris has a stran...,5,31/5/2020 19:36:17,2019-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,750396525,188467,Beautiful and elegant,8FEC7B1C9034428EFF3BF8728B036829,"Lovely architecture, looks different again to ...",4,11/3/2020 06:16:38,2020-03,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2. Preprocessing of the text in the review column


In [15]:
STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Standard Cleaning:
    - lowercase
    - delete URLs
    - delete ponctuation and special chars 
    - delete numbers
    - delete stopwords
    - lemmatization
    - return cleaned text
    """
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # simplified tokenization
    tokens = text.split()

    # stopwords + lemmatization
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOPWORDS and len(t) > 2]

    return " ".join(tokens)

# Apply cleaning
reviews_en["clean_text"] = reviews_en[TEXT_COL].apply(clean_text)

# verify results
display(reviews_en[[PLACE_ID_COL, TEXT_COL, "clean_text"]].head(5))

,idplace,review,clean_text
0,188467,Personally I think it is the most beautiful sq...,personally think beautiful square paris well m...
1,188467,We walked through this lovely park but did not...,walked lovely park stay long square literally ...
2,188467,We come back to this huge square every time we...,come back huge square every time paris spectac...
3,188467,The most beautiful square in Paris has a stran...,beautiful square paris strange name mountain d...
4,188467,"Lovely architecture, looks different again to ...",lovely architecture look different paris squar...


## Step 3: Places representation

1. Group the reviews by place

In [16]:
# Create place documents by concatenating all cleaned reviews for each place
place_docs = (
    reviews_en.groupby(PLACE_ID_COL)["clean_text"]
    .apply(lambda x: " ".join(x.tolist()))
    .reset_index()
    .rename(columns={"clean_text": "place_document"})
)

print("Nb de lieux (docs):", place_docs.shape[0])
display(place_docs.head(5))

Nb de lieux (docs): 1835


,idplace,place_document
0,188467,personally think beautiful square paris well m...
1,188468,old college friend booked beautiful expensive ...
2,188470,winter lot going however easy see cobblestone ...
3,188471,call passe partout shop serious understatement...
4,188472,old historical place attended experience conce...


2. Representation

- Strategy 1: